In [4]:
from __future__ import annotations

from io import StringIO
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

# Make imports/data paths stable whether the notebook is opened from the repo
# root or from notebooks/pitch_count.
CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "src").exists() else CWD.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

YEAR = 2026
TEAMS = [
    "LAA", "HOU", "ATH", "TOR", "ATL", "MIL", "STL", "CHC", "AZ", "LAD",
    "SF", "CLE", "SEA", "MIA", "NYM", "WSH", "BAL", "SD", "PHI", "PIT",
    "TEX", "TB", "BOS", "CIN", "COL", "KC", "DET", "MIN", "CWS", "NYY",
]

REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36"
    ),
    "X-Requested-With": "XMLHttpRequest",
}


In [6]:
SAVANT_CUSTOM_LEADERBOARD_URL = "https://baseballsavant.mlb.com/leaderboard/custom"
SAVANT_BATTER_COUNTING_SELECTIONS = [
    "ab", "pa", "hit", "single", "double", "triple", "home_run", "b_rbi", "r_run",
    "strikeout", "walk", "k_percent", "bb_percent", "batting_avg", "woba", "xwoba",
    "r_total_stolen_base", "r_total_caught_stealing", "sweet_spot_percent",
    "barrel_batted_rate", "hard_hit_percent", "avg_best_speed", "avg_hyper_speed",
    "whiff_percent", "swing_percent",
]
SAVANT_BATTER_COUNTING_PARAMS = {
    "year": YEAR,
    "type": "batter",
    "filter": "",
    "min": "0",
    "selections": ",".join(SAVANT_BATTER_COUNTING_SELECTIONS),
    "chart": "false",
    "x": "ab",
    "y": "ab",
    "r": "no",
    "chartType": "beeswarm",
    "sort": "xwoba",
    "sortDir": "desc",
    "csv": "true",
}
COUNTING_STAT_RENAMES = {
    "last_name, first_name": "player_name",
    "ab": "AB",
    "pa": "PA",
    "hit": "H",
    "single": "1B",
    "double": "2B",
    "triple": "3B",
    "home_run": "HR",
    "b_rbi": "RBI",
    "r_run": "R",
    "strikeout": "SO",
    "walk": "BB",
    "r_total_stolen_base": "SB",
    "r_total_caught_stealing": "CS",
}


def savant_batter_counting_stats_params(year: int = YEAR) -> dict[str, str | int]:
    """Return query params for Savant's grouped batter leaderboard CSV."""
    return SAVANT_BATTER_COUNTING_PARAMS | {"year": year}


def fetch_batter_counting_stats(year: int = YEAR) -> pd.DataFrame:
    """Return the Savant custom batter leaderboard as a DataFrame."""
    response = requests.get(
        SAVANT_CUSTOM_LEADERBOARD_URL,
        params=savant_batter_counting_stats_params(year),
        headers=REQUEST_HEADERS,
        timeout=60,
    )
    response.raise_for_status()

    return pd.read_csv(StringIO(response.text), low_memory=False).rename(columns=COUNTING_STAT_RENAMES)


batter_counting_stats = fetch_batter_counting_stats(YEAR)
batter_counting_stats


,player_name,player_id,year,AB,PA,H,1B,2B,3B,HR,...,xwoba,SB,CS,sweet_spot_percent,barrel_batted_rate,hard_hit_percent,avg_best_speed,avg_hyper_speed,whiff_percent,swing_percent
0,"Young, Jared",676724,2026,308,342,82,53,17,3,9,...,0.321,3,1,30.2,7.7,46.8,101.896709,95.331303,24.4,47.4
1,"Saggese, Thomas",695336,2026,162,180,38,28,5,1,4,...,0.309,4,2,37.1,9.6,43.1,100.738248,94.993078,28.0,51.1
2,"DeLauter, Chase",800050,2026,495,559,142,93,30,3,16,...,0.346,11,2,35.1,6.3,46.3,101.342521,95.083377,15.0,42.3
3,"Muncy, Max",571970,2026,488,561,123,66,27,1,29,...,0.349,3,0,35.8,12.9,45.8,101.023217,94.921257,29.2,44.9
4,"Ballesteros, Moisés",694208,2026,317,352,68,49,10,0,9,...,0.290,1,1,31.0,8.4,45.2,100.762116,94.800705,28.5,51.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
657,"Montgomery, Colson",695657,2026,568,638,112,57,24,0,31,...,0.294,1,1,32.9,12.0,40.9,101.158594,94.756398,37.4,51.5
658,"Clark, Max",703601,2026,180,206,49,32,11,3,3,...,0.314,5,1,35.2,2.1,38.0,98.281387,93.262240,16.7,46.0
659,"McAdoo, Charles",807727,2026,97,106,25,18,4,0,3,...,0.289,2,0,29.7,5.4,25.7,96.549836,92.274918,21.5,49.1
660,"Diaz, Yainer",673237,2026,344,361,98,67,16,1,14,...,0.312,0,2,29.4,6.9,38.8,99.752824,94.074012,22.3,57.4
